# Batch extraction — MyoGait + Sapiens / Sapiens 2

Ce notebook :
1. Installe myogait avec le modèle Sapiens
2. Scanne un dossier (et sous-dossiers) pour trouver toutes les vidéos
3. Lance l'extraction complète (pose + depth + segmentation) sur chaque vidéo
4. Sauvegarde un JSON complet par vidéo au fur et à mesure

Sapiens 2 (ICLR 2026) offre +4 mAP en pose par rapport à v1.
Changer `MODEL` en cell 3 pour basculer entre v1 et v2.

In [ ]:
# ── Cell 1 : Installation ───────────────────────────────────────────
import subprocess, sys

def _run(cmd):
    print(f"▶ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print(f"⚠ stderr: {r.stderr.strip()}")
    return r.returncode

# Installer / mettre à jour myogait
_run(f"{sys.executable} -m pip install --upgrade myogait")

# Dépendances Sapiens (torch + torchvision)
_run(f"{sys.executable} -m pip install --upgrade torch torchvision")

# Dépendances optionnelles pour depth/seg
_run(f"{sys.executable} -m pip install --upgrade opencv-python-headless scipy")

print("\n✔ Installation terminée")

In [ ]:
# ── Cell 2 : Vérification de l'installation ────────────────────────
import myogait as mg
import torch

print(f"myogait  : {mg.__version__}")
print(f"torch    : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Cell 3 : Configuration ─────────────────────────────────────────
from pathlib import Path

# ⬇️  MODIFIER ICI  ⬇️
VIDEO_DIR   = Path("/chemin/vers/les/videos")   # dossier racine des vidéos
OUTPUT_DIR  = Path("/chemin/vers/les/resultats") # dossier de sortie des JSON

# Extensions vidéo reconnues (tout en minuscule, la comparaison est insensible à la casse)
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

# Modèle et options
# Sapiens v1 : "sapiens-quick" (0.3B), "sapiens-mid" (0.6B), "sapiens-top" (1B)
# Sapiens 2  : "sapiens2-quick" (0.4B), "sapiens2-mid" (0.8B), "sapiens2-top" (1B), "sapiens2-ultra" (5B)
MODEL           = "sapiens2-top"  # Sapiens 2 1B — meilleure précision (+4 mAP vs v1)
WITH_DEPTH      = True           # estimation de profondeur
DEPTH_SIZE      = "1b"           # taille du modèle de depth
WITH_SEG        = True           # segmentation corporelle
SEG_SIZE        = "1b"           # taille du modèle de seg
FLIP_IF_RIGHT   = True           # retourner si marche vers la droite
CORRECT_INV     = True           # corriger les inversions L/R
MAX_FRAMES      = None           # None = toutes les frames

print(f"Dossier vidéos  : {VIDEO_DIR}")
print(f"Dossier sortie  : {OUTPUT_DIR}")
print(f"Modèle          : {MODEL}")
print(f"Depth           : {WITH_DEPTH} ({DEPTH_SIZE})")
print(f"Segmentation    : {WITH_SEG} ({SEG_SIZE})")


In [ ]:
# ── Cell 4 : Scan des vidéos ──────────────────────────────────────

video_files = sorted(
    p for p in VIDEO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
)

print(f"Vidéos trouvées : {len(video_files)}\n")
for i, v in enumerate(video_files):
    rel = v.relative_to(VIDEO_DIR)
    print(f"  [{i+1:3d}] {rel}")

if not video_files:
    raise FileNotFoundError(
        f"Aucune vidéo trouvée dans {VIDEO_DIR} "
        f"(extensions: {VIDEO_EXTS})"
    )

In [ ]:
# ── Cell 5 : Extraction en boucle ─────────────────────────────────
import time
import logging
import traceback

# Logging lisible (force=True nécessaire dans Jupyter)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s — %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("batch")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_summary = []   # (video, status, n_frames, duration)
n_total = len(video_files)

for idx, video_path in enumerate(video_files):
    rel = video_path.relative_to(VIDEO_DIR)
    # Garder la structure des sous-dossiers dans la sortie
    out_path = OUTPUT_DIR / rel.with_suffix(".json")

    # Passer si déjà traité
    if out_path.exists():
        log.info("[%d/%d] SKIP (existe) %s", idx + 1, n_total, rel)
        results_summary.append((str(rel), "skipped", 0, 0.0))
        continue

    log.info("[%d/%d] START %s", idx + 1, n_total, rel)
    t0 = time.time()

    try:
        # ── Extraction ──────────────────────────────────────────
        data = mg.extract(
            video_path=str(video_path),
            model=MODEL,
            max_frames=MAX_FRAMES,
            flip_if_right=FLIP_IF_RIGHT,
            correct_inversions=CORRECT_INV,
            with_depth=WITH_DEPTH,
            depth_model_size=DEPTH_SIZE,
            with_seg=WITH_SEG,
            seg_model_size=SEG_SIZE,
            show_progress=True,
        )

        n_frames = len(data.get("frames", []))
        elapsed = time.time() - t0

        # ── Sauvegarde immédiate ────────────────────────────────
        out_path.parent.mkdir(parents=True, exist_ok=True)
        mg.save_json(data, str(out_path))

        log.info(
            "[%d/%d] OK  %s — %d frames in %.1fs (%.1f fps)",
            idx + 1, n_total, rel, n_frames, elapsed,
            n_frames / elapsed if elapsed > 0 else 0,
        )
        results_summary.append((str(rel), "ok", n_frames, elapsed))

    except Exception as exc:
        elapsed = time.time() - t0
        log.error(
            "[%d/%d] FAIL %s — %s", idx + 1, n_total, rel, exc
        )
        traceback.print_exc()
        results_summary.append((str(rel), f"error: {exc}", 0, elapsed))

print("\n" + "=" * 70)
print("BATCH TERMINÉ")
print("=" * 70)

In [ ]:
# ── Cell 6 : Résumé ───────────────────────────────────────────────

n_ok   = sum(1 for _, s, _, _ in results_summary if s == "ok")
n_skip = sum(1 for _, s, _, _ in results_summary if s == "skipped")
n_fail = sum(1 for _, s, _, _ in results_summary if s.startswith("error"))

print(f"Total vidéos  : {n_total}")
print(f"  OK          : {n_ok}")
print(f"  Skipped     : {n_skip}")
print(f"  Erreurs     : {n_fail}")

total_frames = sum(nf for _, s, nf, _ in results_summary if s == "ok")
total_time   = sum(t  for _, s, _, t  in results_summary if s == "ok")

if n_ok > 0:
    print(f"\nFrames totales : {total_frames}")
    print(f"Temps total    : {total_time:.0f}s ({total_time/60:.1f} min)")
    print(f"Vitesse moy.   : {total_frames / total_time:.1f} fps")

if n_fail > 0:
    print("\n⚠ Vidéos en erreur :")
    for name, status, _, _ in results_summary:
        if status.startswith("error"):
            print(f"  • {name} — {status}")